# GANForge -- StyleGAN2 (Standalone Notebook)

**Photorealistic controllable face generation**

This is a **self-contained extract** of the StyleGAN2 section from
the [GANForge Master Notebook](GANForge_Master_Notebook.ipynb) -- every
shared setup utility it needs (imports, device selection, seeding,
`weights_init`, `show_images`, checkpointing, etc.) is included below in
Section 0, so this notebook runs **top to bottom on its own**, in Colab
or locally, without needing any other file.

> **⚠️ Compute note.** See the Section 0 GPU warning below. Heavier
> architectures (Progressive GAN, StyleGAN2, BigGAN, StackGAN, GameGAN)
> include a lightweight from-scratch training loop sized for a single
> GPU (or CPU, slowly) plus, where realistically available, code to load
> strong pretrained weights for inference.

<a id="0"></a>
## 0. Global Setup & Utilities

Everything shared across all twelve GANs lives here: installs, imports,
device selection, seeding, and small helper functions (`weights_init`,
`show_images`, `save_checkpoint`, `Logger`, an `EMA` helper, and a spectral
-norm-aware conv wrapper). Re-using these keeps every later section short
and focused on what makes *that* GAN different.

**🚀 Pro Tip.** In Google Colab: `Runtime -> Change runtime type -> T4 GPU`
before running anything. Everything here auto-detects CUDA and degrades
gracefully to CPU (slow but functional) if none is found.

In [ ]:
# Colab-only: uncomment to install packages not preinstalled in Colab.
# All of these are optional -- every dataset loader below has an offline
# synthetic fallback, so the notebook still runs without them.
# !pip install -q torchaudio datasets soundfile pillow-simd torchmetrics
# !pip install -q pytorch-fid clean-fid


In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import os
import io
import math
import time
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision
import torchvision.transforms as T
import torchvision.utils as vutils

import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except ImportError:
    # Minimal shim so the notebook still runs if tqdm is somehow missing.
    def tqdm(iterable, *args, **kwargs):
        return iterable

warnings.filterwarnings("ignore")  # GAN training is noisy (deprecation spam); keep output readable

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)


In [ ]:
# ---------------------------------------------------------------------------
# Device setup
# ---------------------------------------------------------------------------
# We centralize device selection so every model/tensor in the notebook is
# created consistently. All training loops below check DEVICE.type == "cuda"
# to decide whether to use mixed precision (torch.cuda.amp), which roughly
# halves memory use and often speeds up training 2-3x on modern GPUs.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

if DEVICE.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU found -- running on CPU. Reduce EPOCHS/BATCH_SIZE in "
          "every section for reasonable runtimes.")


In [ ]:
# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
# GANs are notoriously sensitive to initialization -- fixing every RNG makes
# results reproducible across reruns, which matters a lot when you are
# debugging "why did my generator collapse this time".
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [ ]:
# ---------------------------------------------------------------------------
# Shared helper functions used across every GAN section below
# ---------------------------------------------------------------------------

def weights_init(m: nn.Module):
    '''DCGAN-style weight init (Radford et al., 2015).

    Why: GAN training is a saddle-point optimization between two networks
    that constantly move each other's loss landscape. Starting both nets
    from small, zero-centered weights (N(0, 0.02)) avoids one network
    dominating early purely due to init scale, which is a common cause of
    early, unrecoverable mode collapse.

    Uses isinstance (NOT a substring match on the class name) because many
    custom container classes in this notebook -- PGAN_ConvBlock,
    ModulatedConv2d, StyleConvBlock -- have "Conv" in their own name; a
    naive `"Conv" in classname` check would incorrectly try to initialize
    THOSE containers directly (they have no `.weight` of their own) instead
    of only their actual nn.Conv2d/nn.Linear/nn.BatchNorm2d leaf children.
    '''
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if getattr(m, "bias", None) is not None:
            nn.init.constant_(m.bias.data, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)) and getattr(m, "affine", False):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def show_images(images: torch.Tensor, title: str = "", nrow: int = 8, figsize=(8, 8), normalize=True):
    '''Display a batch of image tensors (B, C, H, W) in [-1, 1] or [0, 1] as a grid.'''
    images = images.detach().cpu()
    grid = vutils.make_grid(images, nrow=nrow, normalize=normalize, value_range=(-1, 1) if normalize else None, padding=2)
    plt.figure(figsize=figsize)
    plt.axis("off")
    plt.title(title)
    npimg = grid.numpy()
    if npimg.shape[0] == 1:
        plt.imshow(npimg[0], cmap="gray")
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


def save_checkpoint(state: dict, path: str):
    '''Persist model/optimizer state so long training runs survive a Colab disconnect.'''
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)


def load_checkpoint(path: str, map_location=None):
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=map_location or DEVICE)


class EMA:
    '''Exponential Moving Average of generator weights.

    Why: instantaneous generator weights oscillate a lot during adversarial
    training (they are chasing a moving discriminator). Averaging weights
    over time (as StyleGAN2 and BigGAN both do) gives a much smoother,
    higher-quality generator for *inference*, at zero extra training cost.
    '''
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


class AvgMeter:
    '''Running mean tracker for loss logging.'''
    def __init__(self):
        self.sum, self.n = 0.0, 0

    def update(self, val, k=1):
        self.sum += float(val) * k
        self.n += k

    @property
    def avg(self):
        return self.sum / max(self.n, 1)


def denorm(x: torch.Tensor) -> torch.Tensor:
    '''Map a tensor from [-1, 1] (tanh output range) back to [0, 1] for display/saving.'''
    return (x.clamp(-1, 1) + 1) / 2


CHECKPOINT_DIR = "checkpoints"
SAMPLES_DIR = "samples"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)
print("Shared utilities ready.")


<a id="5"></a>
## 5. StyleGAN (StyleGAN2) -- Photorealistic controllable face generation

### 5.1 Problem Statement & Why this GAN

Karras et al. (2019, 2020) asked a different question than ProGAN: not
just "how do we generate high resolution images stably" but **"how do we
give the user fine, disentangled control over the generated image"**
(e.g. change hairstyle without changing pose or identity). StyleGAN
replaces the "feed noise directly into the network" idea with a **mapping
network** `z -> w` (an intermediate, more disentangled latent space) whose
`w` then **modulates the statistics of every convolution** in a **synthesis
network** that starts from a *learned constant*, not from `z` itself.
StyleGAN2 additionally replaces AdaIN with **weight demodulation** (removes
"droplet" artifacts) and adds **stochastic noise injection** for
fine, non-structural detail (individual hair strands, skin pores). This is
the **photorealistic, controllable generation** problem.

Note this section intentionally does **not** use progressive-growing
training (that was Section 4) -- StyleGAN2 explicitly replaced progressive
growing with a fixed-architecture, skip-connection generator trained
directly at the target resolution, which is what we implement below.

### 5.2 Data Collection & Preparation

In [ ]:
SGAN2_IMG_SIZE = 64          # StyleGAN2 the paper trains up to 1024; we use 64 to fit a single-GPU/CPU demo
SGAN2_Z_DIM = 128
SGAN2_W_DIM = 128
SGAN2_BATCH_SIZE = 32
SGAN2_EPOCHS = 6

sgan2_transform = T.Compose([T.Resize(SGAN2_IMG_SIZE), T.CenterCrop(SGAN2_IMG_SIZE),
                              T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])

def make_synthetic_faces(n=2000, size=SGAN2_IMG_SIZE, seed=SEED):
    '''Procedural "face-like" images (oval head + eyes + mouth, randomized
    color/position/size) used ONLY when CelebA cannot be downloaded, so the
    rest of this section (style mixing, truncation, interpolation) still
    has something meaningful and visually checkable to work with offline.'''
    rng = np.random.default_rng(seed)
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    imgs = np.ones((n, size, size, 3), dtype=np.float32)
    for i in range(n):
        skin = rng.uniform(0.5, 0.95, 3)
        bg = rng.uniform(0.1, 0.6, 3)
        img = np.ones((size, size, 3), dtype=np.float32) * bg
        cx, cy = size / 2 + rng.uniform(-3, 3), size / 2 + rng.uniform(-3, 3)
        rx, ry = size * rng.uniform(0.28, 0.36), size * rng.uniform(0.34, 0.42)
        face = ((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2 <= 1
        img[face] = skin
        eye_dx, eye_dy = rx * 0.45, ry * 0.15
        for sgn in (-1, 1):
            ex, ey = cx + sgn * eye_dx, cy - eye_dy
            eye = (xx - ex) ** 2 + (yy - ey) ** 2 <= (size * 0.035) ** 2
            img[eye] = rng.uniform(0.0, 0.2, 3)
        mouth = (((xx - cx) / (rx * 0.5)) ** 2 + ((yy - (cy + ry * 0.45)) / (ry * 0.12)) ** 2) <= 1
        img[mouth] = rng.uniform(0.3, 0.6, 3) * [1.0, 0.3, 0.3]
        imgs[i] = img
    imgs = torch.from_numpy(imgs).permute(0, 3, 1, 2)  # (N, H, W, C) -> (N, C, H, W)
    return imgs * 2 - 1  # normalize to [-1, 1]

try:
    sgan2_base = torchvision.datasets.CelebA(root="data", split="train", download=True, transform=sgan2_transform)
    sgan2_dataset = torch.utils.data.Subset(sgan2_base, range(min(len(sgan2_base), 20000)))  # cap for fast epochs
    print(f"Loaded real CelebA: using {len(sgan2_dataset)} images")
except Exception as e:
    print(f"CelebA download failed ({e}); falling back to procedurally generated synthetic faces. "
          f"For real face generation, mount/download CelebA-HQ or FFHQ and point sgan2_dataset at it.")
    faces = make_synthetic_faces(2000)
    sgan2_dataset = TensorDataset(faces, torch.zeros(len(faces), dtype=torch.long))

sgan2_loader = DataLoader(sgan2_dataset, batch_size=SGAN2_BATCH_SIZE, shuffle=True, drop_last=True)
real_batch = next(iter(sgan2_loader))[0]
show_images(real_batch[:32], title="StyleGAN2 training data sample", nrow=8)


### 5.3 Model Architecture

In [ ]:
class ModulatedConv2d(nn.Module):
    '''The core StyleGAN2 operation. Instead of AdaIN (normalize activations,
    then scale/shift by style -- StyleGAN1, which produced "droplet"
    artifacts), StyleGAN2 modulates the CONVOLUTION WEIGHTS themselves by
    the style vector `w`, then demodulates (renormalizes) them so each
    output feature map has unit expected variance. Implemented via the
    standard "grouped convolution" trick: build one modulated weight
    tensor PER SAMPLE in the batch, then run a single grouped conv2d.'''
    def __init__(self, in_c, out_c, kernel_size, style_dim, demodulate=True, upsample=False):
        super().__init__()
        self.upsample = upsample
        self.demodulate = demodulate
        self.k = kernel_size
        self.padding = kernel_size // 2
        self.in_c, self.out_c = in_c, out_c
        self.weight = nn.Parameter(torch.randn(out_c, in_c, kernel_size, kernel_size))
        self.style_fc = nn.Linear(style_dim, in_c)   # maps w -> per-input-channel modulation scale
        nn.init.ones_(self.style_fc.bias)             # start modulation near 1.0 (identity) for training stability
        self.scale = 1 / math.sqrt(in_c * kernel_size ** 2)   # equalized learning rate (StyleGAN trick: keep per-layer gradient scale comparable)

    def forward(self, x, w):
        b, in_c, h, wid = x.shape
        if self.upsample:
            x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            h, wid = h * 2, wid * 2
        style = self.style_fc(w).view(b, 1, in_c, 1, 1)               # (b, 1, in_c, 1, 1)
        weight = self.scale * self.weight.unsqueeze(0) * style        # modulate: (b, out_c, in_c, k, k)
        if self.demodulate:
            demod = torch.rsqrt(weight.pow(2).sum(dim=[2, 3, 4], keepdim=True) + 1e-8)
            weight = weight * demod
        weight = weight.view(b * self.out_c, in_c, self.k, self.k)
        x = x.reshape(1, b * in_c, h, wid)
        out = F.conv2d(x, weight, padding=self.padding, groups=b)
        return out.view(b, self.out_c, out.shape[-2], out.shape[-1])


class NoiseInjection(nn.Module):
    '''Per-pixel Gaussian noise, scaled by a LEARNED per-channel weight.
    Why: gives the generator a cheap, stochastic source of fine detail
    (individual hair strands, skin texture) that need not be encoded in
    `w` at all -- this is what lets StyleGAN2 vary micro-texture between
    samples of the *same* w (see Section 5.7).'''
    def __init__(self, channels):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(1, channels, 1, 1))

    def forward(self, x, noise=None):
        if noise is None:
            noise = torch.randn(x.size(0), 1, x.size(2), x.size(3), device=x.device)
        return x + self.weight * noise


class StyleConvBlock(nn.Module):
    def __init__(self, in_c, out_c, style_dim, upsample=False):
        super().__init__()
        self.conv = ModulatedConv2d(in_c, out_c, 3, style_dim, upsample=upsample)
        self.noise = NoiseInjection(out_c)
        self.bias = nn.Parameter(torch.zeros(1, out_c, 1, 1))
        self.act = nn.LeakyReLU(0.2, True)

    def forward(self, x, w):
        x = self.conv(x, w)
        x = self.noise(x)
        return self.act(x + self.bias)


class ToRGB(nn.Module):
    '''Un-demodulated 1x1 modulated conv to 3 channels + additive skip from
    the previous (upsampled) RGB output. Summing RGB contributions from
    EVERY resolution (rather than only the final one) is StyleGAN2's
    "skip generator" design -- it is what let the authors remove
    progressive-growing training while keeping training stable.'''
    def __init__(self, in_c, style_dim):
        super().__init__()
        self.conv = ModulatedConv2d(in_c, 3, 1, style_dim, demodulate=False)
        self.bias = nn.Parameter(torch.zeros(1, 3, 1, 1))

    def forward(self, x, w, skip=None):
        out = self.conv(x, w) + self.bias
        if skip is not None:
            skip = F.interpolate(skip, scale_factor=2, mode="bilinear", align_corners=False)
            out = out + skip
        return out


class StyleGAN2_MappingNetwork(nn.Module):
    '''z -> w. A deep MLP is used (instead of using z directly to modulate
    convs) because the *learned* w-space is empirically far more linear /
    disentangled than the fixed Gaussian z-space -- interpolating in w
    produces much more semantically smooth results (Section 5.7).'''
    def __init__(self, z_dim=SGAN2_Z_DIM, w_dim=SGAN2_W_DIM, n_layers=4):
        super().__init__()
        layers = []
        in_dim = z_dim
        for _ in range(n_layers):
            layers += [nn.Linear(in_dim, w_dim), nn.LeakyReLU(0.2, True)]
            in_dim = w_dim
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        z = z / (z.norm(dim=1, keepdim=True) + 1e-8)   # normalize z before mapping (keeps mapping-net input scale fixed)
        return self.net(z)


SGAN2_RESOLUTIONS = [4, 8, 16, 32, 64]
SGAN2_CHANNELS = {4: 256, 8: 256, 16: 128, 32: 64, 64: 32}


class StyleGAN2_Synthesis(nn.Module):
    '''Builds the image from a LEARNED CONSTANT (not from z!) by repeatedly
    applying style-modulated convolutions and upsampling, summing a toRGB
    contribution at every resolution along the way.'''
    def __init__(self, style_dim=SGAN2_W_DIM, resolutions=SGAN2_RESOLUTIONS, channels=SGAN2_CHANNELS):
        super().__init__()
        self.resolutions = resolutions
        self.const = nn.Parameter(torch.randn(1, channels[resolutions[0]], resolutions[0], resolutions[0]))
        self.first_conv = StyleConvBlock(channels[resolutions[0]], channels[resolutions[0]], style_dim)
        self.first_to_rgb = ToRGB(channels[resolutions[0]], style_dim)
        self.up_blocks = nn.ModuleList()
        self.to_rgbs = nn.ModuleList()
        in_c = channels[resolutions[0]]
        for res in resolutions[1:]:
            out_c = channels[res]
            self.up_blocks.append(StyleConvBlock(in_c, out_c, style_dim, upsample=True))
            self.to_rgbs.append(ToRGB(out_c, style_dim))
            in_c = out_c
        self.num_layers = 1 + len(self.up_blocks)   # number of distinct style inputs consumed (used for style mixing)

    def forward(self, ws):
        '''`ws` is either a single (B, w_dim) tensor (broadcast to every
        layer) or a list of length `num_layers` of (B, w_dim) tensors, one
        per layer -- the latter is what enables STYLE MIXING (Section 5.7):
        using coarse-layer styles from one w and fine-layer styles from
        another.'''
        if torch.is_tensor(ws):
            ws = [ws] * self.num_layers
        b = ws[0].size(0)
        x = self.const.expand(b, -1, -1, -1)
        x = self.first_conv(x, ws[0])
        rgb = self.first_to_rgb(x, ws[0])
        for i, (block, to_rgb) in enumerate(zip(self.up_blocks, self.to_rgbs)):
            x = block(x, ws[i + 1])
            rgb = to_rgb(x, ws[i + 1], skip=rgb)
        return torch.tanh(rgb)


class StyleGAN2_Generator(nn.Module):
    '''Wraps mapping + synthesis, and keeps a running average `w_avg` of
    mapped latents purely for the TRUNCATION TRICK at inference time
    (Section 5.6): interpolating w toward its mean trades sample diversity
    for sample quality/typicality, with a single scalar knob.'''
    def __init__(self):
        super().__init__()
        self.mapping = StyleGAN2_MappingNetwork()
        self.synthesis = StyleGAN2_Synthesis()
        self.register_buffer("w_avg", torch.zeros(SGAN2_W_DIM))

    def forward(self, z, truncation: float = 1.0, mixing_w2: torch.Tensor = None, mixing_layer: int = None):
        w = self.mapping(z)
        with torch.no_grad():
            self.w_avg.mul_(0.995).add_(w.mean(0).detach(), alpha=0.005)   # cheap running-mean EMA of w
        if truncation < 1.0:
            w = self.w_avg.unsqueeze(0) + truncation * (w - self.w_avg.unsqueeze(0))
        if mixing_w2 is not None:
            ws = [w] * mixing_layer + [mixing_w2] * (self.synthesis.num_layers - mixing_layer)
            return self.synthesis(ws)
        return self.synthesis(w)


class StyleGAN2_DiscBlock(nn.Module):
    '''Residual downsampling block (StyleGAN2's discriminator is a ResNet).
    The 1x1 skip path (scaled by 1/sqrt(2) to preserve variance) lets
    gradients skip the two 3x3 convs, which the paper found meaningfully
    stabilizes training of deeper discriminators.'''
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, in_c, 3, 1, 1)
        self.conv2 = nn.Conv2d(in_c, out_c, 3, 1, 1)
        self.skip = nn.Conv2d(in_c, out_c, 1, bias=False)
        self.act = nn.LeakyReLU(0.2, True)

    def forward(self, x):
        y = self.act(self.conv1(x))
        y = self.act(self.conv2(y))
        y = F.avg_pool2d(y, 2)
        skip = F.avg_pool2d(self.skip(x), 2)
        return (y + skip) / math.sqrt(2)


class StyleGAN2_Discriminator(nn.Module):
    def __init__(self, resolutions=SGAN2_RESOLUTIONS, channels=SGAN2_CHANNELS):
        super().__init__()
        rev = list(reversed(resolutions))
        self.from_rgb = nn.Conv2d(3, channels[rev[0]], 1)
        blocks = []
        for i in range(len(rev) - 1):
            blocks.append(StyleGAN2_DiscBlock(channels[rev[i]], channels[rev[i + 1]]))
        self.blocks = nn.Sequential(*blocks)
        final_c = channels[rev[-1]]
        # Minibatch standard deviation: append one extra feature map holding
        # the std-dev of features ACROSS the batch -- gives D a cheap signal
        # for "does this whole batch look too similar" (a direct anti-mode-
        # collapse mechanism from the ProGAN/StyleGAN family).
        self.final_conv = nn.Conv2d(final_c + 1, final_c, 3, 1, 1)
        self.final_fc = nn.Sequential(nn.Flatten(), nn.Linear(final_c * rev[-1] * rev[-1], final_c), nn.LeakyReLU(0.2, True), nn.Linear(final_c, 1))

    def minibatch_stddev(self, x):
        std = x.std(dim=0, unbiased=False).mean().expand(x.size(0), 1, x.size(2), x.size(3))
        return torch.cat([x, std], dim=1)

    def forward(self, x):
        x = F.leaky_relu(self.from_rgb(x), 0.2)
        x = self.blocks(x)
        x = self.minibatch_stddev(x)
        x = F.leaky_relu(self.final_conv(x), 0.2)
        return self.final_fc(x).view(-1)


G_sgan2 = StyleGAN2_Generator().to(DEVICE).apply(weights_init)
D_sgan2 = StyleGAN2_Discriminator().to(DEVICE).apply(weights_init)
print(f"StyleGAN2-lite: {sum(p.numel() for p in G_sgan2.parameters())/1e6:.1f}M generator params, "
      f"{sum(p.numel() for p in D_sgan2.parameters())/1e6:.1f}M discriminator params")


### 5.4 Loss Functions & Optimizers

In [ ]:
sgan2_criterion = nn.BCEWithLogitsLoss()
sgan2_opt_G = torch.optim.Adam(G_sgan2.parameters(), lr=2e-4, betas=(0.0, 0.99))
sgan2_opt_D = torch.optim.Adam(D_sgan2.parameters(), lr=2e-4, betas=(0.0, 0.99))

SGAN2_R1_GAMMA = 5.0
SGAN2_R1_EVERY = 16   # "lazy" regularization: StyleGAN2 only applies R1 every N steps to save compute, scaling its weight by N to compensate

def sgan2_r1_penalty(D, real):
    '''R1 gradient penalty (Mescheder et al. / StyleGAN2): penalize the
    NORM of D's gradient w.r.t. REAL images only (unlike WGAN-GP's penalty
    on an interpolated point). This directly discourages D from having a
    sharp decision boundary right at the real-data manifold, which is
    empirically what actually causes non-convergent oscillation in
    practice -- and it is cheaper than WGAN-GP since it needs no fake pass.'''
    real = real.requires_grad_(True)
    logits = D(real)
    grad = torch.autograd.grad(outputs=logits.sum(), inputs=real, create_graph=True)[0]
    return grad.pow(2).reshape(grad.size(0), -1).sum(1).mean()


### 5.5 Training Loop

In [ ]:
# ---------------------------------------------------------------------------
# 5.5a Lightweight from-scratch training
# ---------------------------------------------------------------------------
sgan2_fixed_z = torch.randn(16, SGAN2_Z_DIM, device=DEVICE)
sgan2_step = 0
sgan2_history = []

for epoch in range(SGAN2_EPOCHS):
    meters = {"G": AvgMeter(), "D": AvgMeter()}
    pbar = tqdm(sgan2_loader, desc=f"[StyleGAN2] epoch {epoch+1}/{SGAN2_EPOCHS}")
    for real, *_ in pbar:
        real = real.to(DEVICE)
        b = real.size(0)
        sgan2_step += 1

        # ---- Discriminator ----
        D_sgan2.zero_grad(set_to_none=True)
        z = torch.randn(b, SGAN2_Z_DIM, device=DEVICE)
        fake = G_sgan2(z)
        logits_real = D_sgan2(real)
        logits_fake = D_sgan2(fake.detach())
        loss_d = sgan2_criterion(logits_real, torch.ones(b, device=DEVICE)) + \
                 sgan2_criterion(logits_fake, torch.zeros(b, device=DEVICE))
        if sgan2_step % SGAN2_R1_EVERY == 0:
            r1 = sgan2_r1_penalty(D_sgan2, real)
            loss_d = loss_d + SGAN2_R1_GAMMA / 2 * r1 * SGAN2_R1_EVERY
        loss_d.backward()
        sgan2_opt_D.step()

        # ---- Generator ----
        G_sgan2.zero_grad(set_to_none=True)
        logits_fake_g = D_sgan2(fake)
        loss_g = sgan2_criterion(logits_fake_g, torch.ones(b, device=DEVICE))
        loss_g.backward()
        sgan2_opt_G.step()

        meters["D"].update(loss_d.item()); meters["G"].update(loss_g.item())
        pbar.set_postfix(loss_D=f"{loss_d.item():.3f}", loss_G=f"{loss_g.item():.3f}")

    sgan2_history.append((meters["G"].avg, meters["D"].avg))
    with torch.no_grad():
        G_sgan2.eval()
        sample = G_sgan2(sgan2_fixed_z, truncation=0.7)
        G_sgan2.train()
    show_images(sample, title=f"StyleGAN2-lite samples after epoch {epoch+1}", nrow=4)
    save_checkpoint({"G": G_sgan2.state_dict(), "D": D_sgan2.state_dict()}, f"{CHECKPOINT_DIR}/sgan2_epoch{epoch+1}.pt")

print("StyleGAN2-lite training complete.")


In [ ]:
# ---------------------------------------------------------------------------
# 5.5b Loading strong pretrained weights (inference-only mode)
# ---------------------------------------------------------------------------
# Official StyleGAN2/StyleGAN2-ADA checkpoints are distributed as .pkl files
# whose loading code (legacy.py / dnnlib) lives in NVIDIA's own repo, which
# also ships custom CUDA ops (upfirdn2d, fused bias-act) for full speed.
# We attempt to clone the repo and load an official FFHQ checkpoint;
# without network access this cleanly falls back to the from-scratch model.
def try_load_pretrained_stylegan2(pkl_url="https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl"):
    try:
        import subprocess, sys
        if not os.path.isdir("stylegan2-ada-pytorch"):
            subprocess.run(["git", "clone", "--depth", "1",
                             "https://github.com/NVlabs/stylegan2-ada-pytorch.git"], check=True)
        sys.path.insert(0, "stylegan2-ada-pytorch")
        import legacy, dnnlib  # provided by the cloned repo
        with dnnlib.util.open_url(pkl_url) as f:
            G_ema = legacy.load_network_pkl(f)["G_ema"].to(DEVICE).eval()
        print("Loaded official pretrained StyleGAN2-ADA generator (FFHQ).")
        return G_ema
    except Exception as e:
        print(f"Pretrained StyleGAN2 unavailable in this environment ({e}). "
              f"Falling back to the from-scratch StyleGAN2-lite model trained above.")
        return None

sgan2_pretrained = try_load_pretrained_stylegan2()


### 5.6 Inference / Generation Function

In [ ]:
@torch.no_grad()
def sgan2_generate(n_samples: int = 16, truncation: float = 0.7, seed: int = None, use_pretrained: bool = False) -> torch.Tensor:
    '''Truncation trick: truncation=1.0 -> full diversity (may include odd
    samples); truncation<1.0 pulls w toward the running average w_avg,
    trading diversity for consistently "typical", higher-quality faces.'''
    if seed is not None:
        torch.manual_seed(seed)
    if use_pretrained and sgan2_pretrained is not None:
        z = torch.randn(n_samples, sgan2_pretrained.z_dim, device=DEVICE)
        return sgan2_pretrained(z, None, truncation_psi=truncation)
    G_sgan2.eval()
    z = torch.randn(n_samples, SGAN2_Z_DIM, device=DEVICE)
    imgs = G_sgan2(z, truncation=truncation)
    G_sgan2.train()
    return imgs

show_images(sgan2_generate(16, truncation=0.7), title="StyleGAN2-lite inference (truncation=0.7)", nrow=4)


### 5.7 Visualization & Results

In [ ]:
gs, ds = zip(*sgan2_history)
plt.figure(figsize=(6, 4)); plt.plot(gs, label="G"); plt.plot(ds, label="D")
plt.legend(); plt.title("StyleGAN2-lite training losses"); plt.show()

# Truncation sweep: same z's, increasing truncation -- shows the
# diversity/quality trade-off directly.
with torch.no_grad():
    G_sgan2.eval()
    z_fixed = torch.randn(8, SGAN2_Z_DIM, device=DEVICE)
    for psi in [0.2, 0.5, 1.0]:
        imgs = G_sgan2(z_fixed, truncation=psi)
        show_images(imgs, title=f"Truncation psi={psi}", nrow=8, figsize=(12, 2))

    # Style mixing: coarse layers from w1 (pose/shape), fine layers from w2
    # (color/texture) -- the signature StyleGAN capability that vanilla
    # GAN latent space does not offer, because early layers control
    # low-resolution structure and late layers control high-resolution
    # detail almost independently in w-space.
    z1, z2 = torch.randn(4, SGAN2_Z_DIM, device=DEVICE), torch.randn(4, SGAN2_Z_DIM, device=DEVICE)
    w2 = G_sgan2.mapping(z2)
    mixed = G_sgan2(z1, mixing_w2=w2, mixing_layer=2)   # layers 0-1 from z1, layers 2+ from z2
    show_images(mixed, title="Style mixing: coarse=z1, fine=z2", nrow=4, figsize=(8, 3))
    G_sgan2.train()

# Latent interpolation in W-space (smoother than Z-space -- see 5.1).
@torch.no_grad()
def sgan2_interpolate_w(n_steps=8):
    G_sgan2.eval()
    z1, z2 = torch.randn(1, SGAN2_Z_DIM, device=DEVICE), torch.randn(1, SGAN2_Z_DIM, device=DEVICE)
    w1, w2 = G_sgan2.mapping(z1), G_sgan2.mapping(z2)
    alphas = torch.linspace(0, 1, n_steps, device=DEVICE).view(-1, 1)
    w_interp = (1 - alphas) * w1 + alphas * w2
    imgs = G_sgan2.synthesis(w_interp)
    G_sgan2.train()
    return imgs

show_images(sgan2_interpolate_w(8), title="W-space interpolation", nrow=8, figsize=(12, 2))


### 5.8 User Input → Output Demo

In [ ]:
N_SAMPLES_TO_GENERATE = 9
TRUNCATION_PSI = 0.6   # lower = more "average"/safe faces, higher = more diverse/unusual
RANDOM_SEED = 99

demo = sgan2_generate(N_SAMPLES_TO_GENERATE, truncation=TRUNCATION_PSI, seed=RANDOM_SEED)
show_images(demo, title=f"User demo: truncation={TRUNCATION_PSI}", nrow=3)


### 5.9 Key Insights & Limitations of this GAN

- 💡 **Insight.** Weight demodulation is a *deterministic, statistics-based*
  approximation of "normalize then modulate" (AdaIN) that never actually
  looks at the real activation statistics -- this decoupling from
  instance-specific statistics is precisely what removes StyleGAN1's
  characteristic droplet artifacts.
- 💡 **Insight.** Style mixing regularization (training with mixed w's, not
  just evaluating with them) is what makes the coarse/fine layer split
  semantically meaningful in the first place -- without it, mixing tends
  to just look broken.
- ⚠️ **Warning.** Our 64x64 lightweight model captures the *mechanism*
  (mapping network, modulated convs, noise injection, truncation, style
  mixing) but not the photorealism of the 1024x1024 FFHQ-trained original,
  which needs orders of magnitude more data/compute -- use
  `try_load_pretrained_stylegan2()` above for real photorealistic results.

**This GAN solved the problem of: photorealistic generation with
disentangled, user-controllable style -- separating "what identity/pose"
from "what fine texture/detail" in a way plain latent-space GANs cannot.**